# Эмоции: обычное лицо, радость и грусть

Transfer learning на EfficientNetB0. Используются только три сбалансированных класса AffectNet, а общий лимит обучения — **28 минут**. Расчёт ориентирован на Colab с GPU T4; на другом оборудовании время и качество могут отличаться.


In [ ]:
!pip -q install tensorflow

import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "не найден — обучение будет медленнее")


In [ ]:
# Нужен kaggle.json (Kaggle → Profile → Account → API → Create New Token).
# Положите его в корень Google Drive ИЛИ загрузите через диалог ниже.
from google.colab import drive

drive.mount('/content/drive/')

# if not os.path.exists('/root/.kaggle/kaggle.json'):
#     if os.path.exists('/content/drive/MyDrive/kaggle.json'):
#         !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#     else:
#         from google.colab import files
#         print('Выберите скачанный kaggle.json:')
#         uploaded = files.upload()
#         !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

AFFECTNET_DIR = Path('/content/affectnet')
if not (AFFECTNET_DIR / 'train').exists():
    KAGGLE_DS = 'chups1509/affectnet'
    !kaggle datasets download -d $KAGGLE_DS -p /content/affectnet_zip
    !unzip -q -o /content/affectnet_zip/*.zip -d /content/affectnet
    !rm -rf /content/affectnet_zip

print('Датасет:', AFFECTNET_DIR)


In [ ]:
# Стандартная разметка AffectNet: 0 — neutral, 1 — happy, 2 — sad.
SOURCE_TO_NAME = {'0': 'neutral', '1': 'happy', '2': 'sad'}
class_names = ['neutral', 'happy', 'sad']
display_names = {
    'neutral': 'Обычное лицо',
    'happy': 'Радость',
    'sad': 'Грусть',
}

MAX_TRAIN_PER_CLASS = 5000
IMG_SIZE = (96, 96)
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE
DATA_DIR = Path('/content/affectnet_3class')
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Делаем небольшой трёхклассовый набор через символические ссылки:
# исходные изображения не копируются и не изменяются.
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

rng = random.Random(SEED)
train_files = {}
for source_label, target_name in SOURCE_TO_NAME.items():
    folder = AFFECTNET_DIR / 'train' / source_label
    files = [p for p in folder.iterdir() if p.suffix.lower() in EXTENSIONS]
    if not files:
        raise FileNotFoundError(f'Нет изображений в {folder}')
    train_files[target_name] = files

# Одинаковое число примеров каждого класса уменьшает перекос предсказаний.
train_per_class = min(
    MAX_TRAIN_PER_CLASS,
    min(len(files) for files in train_files.values()),
)

for split in ['train', 'val', 'test']:
    for source_label, target_name in SOURCE_TO_NAME.items():
        source_folder = AFFECTNET_DIR / split / source_label
        files = [p for p in source_folder.iterdir() if p.suffix.lower() in EXTENSIONS]
        rng.shuffle(files)
        if split == 'train':
            files = files[:train_per_class]
        target_folder = DATA_DIR / split / target_name
        target_folder.mkdir(parents=True, exist_ok=True)
        for source in files:
            (target_folder / source.name).symlink_to(source.resolve())
        print(f'{split:5s} | {target_name:7s}: {len(files)}')

def load_split(split, shuffle=False):
    return keras.utils.image_dataset_from_directory(
        DATA_DIR / split,
        class_names=class_names,
        image_size=IMG_SIZE,
        batch_size=BATCH,
        label_mode='int',
        shuffle=shuffle,
        seed=SEED,
    )

train_ds = load_split('train', shuffle=True)
val_ds = load_split('val')
test_ds = load_split('test')

augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.05),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomContrast(0.1),
], name='augmentation')

train_ds = train_ds.map(
    lambda x, y: (augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


In [ ]:
# EfficientNetB0 уже содержит нужную предобработку, поэтому вход — RGB 0…255.
base = keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=IMG_SIZE + (3,),
    pooling='avg',
)
base.trainable = False

model = keras.Sequential([
    base,
    keras.layers.Dropout(0.25),
    keras.layers.Dense(len(class_names), activation='softmax'),
], name='emotion_3class')

class TimeBudget(keras.callbacks.Callback):
    """Останавливает обе фазы по общему лимиту, не дожидаясь конца эпохи."""
    def __init__(self, minutes=28):
        super().__init__()
        self.limit_seconds = minutes * 60
        self.started_at = None
        self.expired = False

    @property
    def elapsed_seconds(self):
        return 0 if self.started_at is None else time.monotonic() - self.started_at

    def on_train_begin(self, logs=None):
        if self.started_at is None:
            self.started_at = time.monotonic()

    def on_train_batch_end(self, batch, logs=None):
        if self.elapsed_seconds >= self.limit_seconds:
            self.expired = True
            self.model.stop_training = True
            print(f'\nДостигнут лимит {self.limit_seconds / 60:.0f} минут.')

checkpoint_dir = Path('/content/checkpoints')
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = checkpoint_dir / 'emotion_3class_best.weights.h5'
checkpoint = keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True,
)
time_budget = TimeBudget(minutes=28)

# Фаза 1: быстро обучаем только классификатор.
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
h1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=4,
    callbacks=[checkpoint, time_budget],
)
histories = [h1.history]

# Фаза 2: дообучаем верхние слои для более точного различения эмоций.
if not time_budget.expired:
    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False
    # BatchNorm оставляем замороженным: так обучение стабильнее на небольшом наборе.
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(2e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    )
    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=[checkpoint, early_stop, time_budget],
    )
    histories.append(h2.history)

# Загружаем лучшие веса по val_loss за обе фазы, а не просто последние.
if checkpoint_path.exists():
    model.load_weights(checkpoint_path)

history = keras.callbacks.History()
history.history = {
    key: [value for part in histories for value in part.get(key, [])]
    for key in histories[0]
}

model.save('/content/drive/MyDrive/emotion_3class_model.keras')
print(f'Обучение заняло {time_budget.elapsed_seconds / 60:.1f} мин.')
print('Лучшая модель сохранена: /content/drive/MyDrive/emotion_3class_model.keras')


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Валидация: accuracy={val_acc:.3f}, loss={val_loss:.3f}')
print(f'Тест:      accuracy={test_acc:.3f}, loss={test_loss:.3f}')

# Матрица ошибок и точность каждого класса на отложенном тесте.
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_pred = np.concatenate([
    np.argmax(model.predict(images, verbose=0), axis=1)
    for images, _ in test_ds
])
confusion = tf.math.confusion_matrix(y_true, y_pred, num_classes=3).numpy()
for i, name in enumerate(class_names):
    class_acc = confusion[i, i] / max(confusion[i].sum(), 1)
    print(f'{display_names[name]:13s}: {class_acc:.3f}')

plt.figure(figsize=(5, 4))
plt.imshow(confusion, cmap='Blues')
plt.title('Матрица ошибок (тест)')
plt.xticks(range(3), [display_names[n] for n in class_names], rotation=20)
plt.yticks(range(3), [display_names[n] for n in class_names])
plt.xlabel('Предсказание')
plt.ylabel('Истина')
for i in range(3):
    for j in range(3):
        plt.text(j, i, confusion[i, j], ha='center', va='center')
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
val_shuffled = val_ds.shuffle(200, seed=42, reshuffle_each_iteration=False)
images, labels = next(iter(val_shuffled))
probs = model.predict(images, verbose=0)

plt.figure(figsize=(12, 8))
for i in range(min(9, len(images))):
    pred = np.argmax(probs[i])
    true = labels[i].numpy()
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype('uint8'))
    plt.axis('off')
    color = 'green' if pred == true else 'red'
    pred_name = display_names[class_names[pred]]
    true_name = display_names[class_names[true]]
    plt.title(
        f'Пред: {pred_name} ({probs[i][pred]:.0%})\nИстина: {true_name}',
        color=color,
        fontsize=9,
    )
plt.tight_layout()
plt.show()
